# 02 — Baseline Training: EfficientNet-B3 on CIFAKE

This notebook:
1. Sets seeds and detects GPU
2. Builds the EfficientNet-B3 detector and loads CIFAKE
3. **Runs a hyperparameter sweep** (learning rate × weight decay) over 3 quick epochs
4. Configures the best hyperparameters for full training
5. Runs 20-epoch training loop with checkpointing
6. Plots training curves
7. Evaluates on the test set (accuracy, AUC, ECE)

**Prerequisites:** Run `01_setup_and_eda.ipynb` first to download CIFAKE.

In [2]:
# Cell 1 — Mount Google Drive & define paths
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
REPO_DIR = "/content/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone the repo (update URL to your own fork)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Data directory:    {DATA_DIR}")
print(f"Checkpoint dir:    {CHECKPOINT_DIR}")

Mounted at /content/drive
Cloning into '/content/ai-image-detection'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 44 (delta 10), reused 40 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 347.04 KiB | 10.52 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Working directory: /content/ai-image-detection
Data directory:    /content/ai-image-detection/data/raw/cifake
Checkpoint dir:    /content/drive/MyDrive/ai-image-detection/checkpoints


In [3]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 73.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.5 MB/s et

In [4]:
# Cell 3 — Extract CIFAKE from zip uploaded to Google Drive
import os, zipfile, glob

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(os.path.join(DATA_DIR, "train")):
    print("CIFAKE already extracted — skipping.")
else:
    # Find the zip file in the Drive folder (handles any filename)
    zip_candidates = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            f"No zip file found in {DRIVE_ROOT}. "
            "Upload the CIFAKE zip downloaded from Kaggle to that folder."
        )

    zip_path = zip_candidates[0]
    print(f"Found zip: {zip_path}")
    print("Extracting (this takes ~1-2 minutes)...")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)

    print("Extraction complete.")

# Show what was extracted so we can verify the folder structure
extracted = os.listdir(DATA_DIR)
print(f"Contents of DATA_DIR: {extracted}")

# If zip extracted into a nested subfolder, adjust DATA_DIR automatically
if "train" not in extracted and len(extracted) == 1:
    DATA_DIR = os.path.join(DATA_DIR, extracted[0])
    print(f"Adjusted DATA_DIR to: {DATA_DIR}")

print(f"Final DATA_DIR contents: {os.listdir(DATA_DIR)}")

Found zip: /content/drive/MyDrive/ai-image-detection/archive (3).zip
Extracting (this takes ~1-2 minutes)...
Extraction complete.
Contents of DATA_DIR: ['train', 'test']
Final DATA_DIR contents: ['train', 'test']


In [5]:
# Cell 4 — Verify dataset splits
import os

splits = {}
for split in ["train", "test"]:
    for cls in ["REAL", "FAKE"]:
        path = os.path.join(DATA_DIR, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        splits[f"{split}/{cls}"] = count

print("CIFAKE Dataset Summary")
print("=" * 30)
for key, count in splits.items():
    print(f"  {key:15s} : {count:,}")
print(f"  {'Total':15s} : {sum(splits.values()):,}")

assert splits["train/REAL"] == 50_000 or splits["train/REAL"] == 30_000, (
    f"Unexpected train/REAL count: {splits['train/REAL']}"
)

CIFAKE Dataset Summary
  train/REAL      : 50,000
  train/FAKE      : 50,000
  test/REAL       : 10,000
  test/FAKE       : 10,000
  Total           : 120,000


In [6]:
# Cell 1 — Imports, seeds, device
import os, sys
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

REPO_DIR = "/content/ai-image-detection"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
PLOTS_DIR = os.path.join(REPO_DIR, "outputs", "plots")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [7]:
# Cell 2 — Build model & load data
from src.model.detector import build_detector
from src.utils.data_loader import get_cifake_loaders

model = build_detector(pretrained=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

train_loader, val_loader, test_loader = get_cifake_loaders(
    DATA_DIR, batch_size=32, num_workers=2,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Total parameters:     10,699,306
Trainable parameters: 10,699,306


## Hyperparameter Tuning

Before committing to a full 20-epoch training run, we search over a small grid of learning rates and weight decay values. Each combination is trained for **3 epochs** on the training split and evaluated on the validation split. The combination with the highest validation accuracy is selected for the full run.

**Search space:**
- Learning rate: `[3e-4, 1e-4, 5e-5]`
- Weight decay: `[1e-4, 1e-5]`

This gives 6 combinations. Each combo is in its **own cell** (~45 min each on a T4 GPU). Results are saved to Google Drive after each combo, so if Colab disconnects you can **resume** without re-running completed combos — just re-run the setup cells and continue from where you left off.

In [ ]:
# Cell 3a — Sweep setup: resume loader + helper function
import json

SWEEP_EPOCHS = 3
SWEEP_RESULTS_PATH = os.path.join(DRIVE_ROOT, "sweep_results.json")

# Resume: load any previously completed combos from Drive
if os.path.exists(SWEEP_RESULTS_PATH):
    with open(SWEEP_RESULTS_PATH) as f:
        sweep_results = json.load(f)
    print(f"Resumed sweep — {len(sweep_results)} combo(s) already done:")
    for r in sweep_results:
        print(f"  lr={r['lr']:.0e}  wd={r['wd']:.0e}  -> val_acc={r['val_acc']:.4f}")
else:
    sweep_results = []
    print("Starting fresh sweep (no previous results on Drive).")

done_keys = {(r["lr"], r["wd"]) for r in sweep_results}


def run_one_combo(lr, wd):
    """Train one lr/wd combo for SWEEP_EPOCHS and save result to Drive."""
    if (lr, wd) in done_keys:
        print(f"  lr={lr:.0e}  wd={wd:.0e}  -> already done, skipping")
        return

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    sweep_model = build_detector(pretrained=True).to(device)
    sweep_optimizer = torch.optim.AdamW(sweep_model.parameters(), lr=lr, weight_decay=wd)
    sweep_criterion = nn.CrossEntropyLoss()

    best_val = 0.0
    for ep in range(1, SWEEP_EPOCHS + 1):
        sweep_model.train()
        for images, labels in tqdm(train_loader, leave=False, desc=f"lr={lr} wd={wd} ep={ep}"):
            images, labels = images.to(device), labels.to(device)
            sweep_optimizer.zero_grad()
            loss = sweep_criterion(sweep_model(images), labels)
            loss.backward()
            sweep_optimizer.step()

        sweep_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                correct += (sweep_model(images).argmax(1) == labels).sum().item()
                total += images.size(0)
        val_acc = correct / total
        best_val = max(best_val, val_acc)

    sweep_results.append({"lr": lr, "wd": wd, "val_acc": best_val})
    done_keys.add((lr, wd))
    print(f"  lr={lr:.0e}  wd={wd:.0e}  -> best val_acc={best_val:.4f}")

    # Save to Drive after each combo so progress survives disconnects
    with open(SWEEP_RESULTS_PATH, "w") as f:
        json.dump(sweep_results, f, indent=2)
    print(f"  (saved to {SWEEP_RESULTS_PATH})")

    del sweep_model, sweep_optimizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


print(f"\nSweep helper ready. Run the next 6 cells one at a time.")

lr=0.0003 wd=0.0001 ep=1:   0%|          | 0/2500 [00:00<?, ?it/s]

lr=0.0003 wd=0.0001 ep=2:   0%|          | 0/2500 [00:00<?, ?it/s]

lr=0.0003 wd=0.0001 ep=3:   0%|          | 0/2500 [00:00<?, ?it/s]

  lr=3e-04  wd=1e-04  -> best val_acc=0.9594


lr=0.0003 wd=1e-05 ep=1:   0%|          | 0/2500 [00:00<?, ?it/s]

lr=0.0003 wd=1e-05 ep=2:   0%|          | 0/2500 [00:00<?, ?it/s]

lr=0.0003 wd=1e-05 ep=3:   0%|          | 0/2500 [00:00<?, ?it/s]

  lr=3e-04  wd=1e-05  -> best val_acc=0.9515


lr=0.0001 wd=0.0001 ep=1:   0%|          | 0/2500 [00:00<?, ?it/s]

lr=0.0001 wd=0.0001 ep=2:   0%|          | 0/2500 [00:00<?, ?it/s]

In [ ]:
# Combo 1 of 6: lr=3e-4, wd=1e-4
run_one_combo(lr=3e-4, wd=1e-4)

In [ ]:
# Combo 2 of 6: lr=3e-4, wd=1e-5
run_one_combo(lr=3e-4, wd=1e-5)

In [ ]:
# Combo 3 of 6: lr=1e-4, wd=1e-4
run_one_combo(lr=1e-4, wd=1e-4)

In [ ]:
# Combo 4 of 6: lr=1e-4, wd=1e-5
run_one_combo(lr=1e-4, wd=1e-5)

In [ ]:
# Combo 5 of 6: lr=5e-5, wd=1e-4
run_one_combo(lr=5e-5, wd=1e-4)

In [ ]:
# Combo 6 of 6: lr=5e-5, wd=1e-5
run_one_combo(lr=5e-5, wd=1e-5)

In [ ]:
# Cell 3b — Pick best hyperparameters from sweep
sweep_results.sort(key=lambda x: x["val_acc"], reverse=True)
best_hp = sweep_results[0]

print(f"{'='*50}")
print(f"  Hyperparameter Sweep Results")
print(f"{'='*50}")
for r in sweep_results:
    marker = " <-- BEST" if r["lr"] == best_hp["lr"] and r["wd"] == best_hp["wd"] else ""
    print(f"  lr={r['lr']:.0e}  wd={r['wd']:.0e}  val_acc={r['val_acc']:.4f}{marker}")
print(f"{'='*50}")
print(f"\nSelected: lr={best_hp['lr']:.0e}, wd={best_hp['wd']:.0e}")

In [ ]:
# Cell 3c — Configure full training with best hyperparameters
NUM_EPOCHS = 20
LR = best_hp["lr"]
WEIGHT_DECAY = best_hp["wd"]

# Re-initialise model with fresh pretrained weights for the full run
model = build_detector(pretrained=True).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()

print(f"Optimizer:  AdamW (lr={LR:.0e}, weight_decay={WEIGHT_DECAY:.0e})")
print(f"Scheduler:  CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"Loss:       CrossEntropyLoss")
print(f"Epochs:     {NUM_EPOCHS}")
print(f"(Hyperparameters selected from sweep above)")

In [ ]:
# Cell 4a — Training setup (checks for resume checkpoint on Drive)
from src.model.detector import save_checkpoint

best_checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_detector.pth")
resume_checkpoint_path = os.path.join(CHECKPOINT_DIR, "resume_checkpoint.pth")

# Check for a resume checkpoint from a previous interrupted session
if os.path.exists(resume_checkpoint_path):
    resume_ckpt = torch.load(resume_checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(resume_ckpt["model_state_dict"])
    optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])
    history = resume_ckpt["history"]
    best_val_acc = resume_ckpt["best_val_acc"]
    start_epoch = resume_ckpt["epoch"] + 1
    print(f"Resumed from epoch {resume_ckpt['epoch']} (best_val_acc={best_val_acc:.4f})")
    print(f"Continuing from epoch {start_epoch} to {NUM_EPOCHS}")
else:
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    start_epoch = 1
    print(f"Starting fresh training from epoch 1 to {NUM_EPOCHS}")


def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.size(0)

    return running_loss / total, correct / total


print("Training setup ready. Run the next cell to start/resume training.")

In [ ]:
# Cell 4b — Resumable training loop (saves full state to Drive after every epoch)
for epoch in range(start_epoch, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    lr_now = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
        f"LR: {lr_now:.6f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        metrics = {"val_acc": val_acc, "val_loss": val_loss, "train_acc": train_acc}
        save_checkpoint(model, best_checkpoint_path, epoch, metrics)
        print(f"  -> Saved best checkpoint (val_acc={val_acc:.4f})")

    # Save full training state to Drive for resume after every epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_val_acc": best_val_acc,
    }, resume_checkpoint_path)

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
# Cell 4c — Cleanup: remove resume checkpoint (training is done)
if os.path.exists(resume_checkpoint_path):
    os.remove(resume_checkpoint_path)
    print("Removed resume checkpoint (training complete).")
else:
    print("No resume checkpoint to remove.")

print(f"Final best val accuracy: {best_val_acc:.4f}")
print(f"Best model saved at: {best_checkpoint_path}")

In [ ]:
# Cell 5 — Plot training curves
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs, history["train_acc"], label="Train")
ax2.plot(epochs, history["val_acc"], label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()

fig.suptitle("EfficientNet-B3 Training on CIFAKE", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "training_curves.png"), dpi=150)
plt.show()
print(f"Saved to {PLOTS_DIR}/training_curves.png")

In [ ]:
# Cell 6 — Test set evaluation
from src.model.detector import load_checkpoint
from src.evaluation.metrics import (
    compute_accuracy, compute_auc, compute_ece,
    plot_reliability_diagram, plot_roc_curve,
)

# Load best checkpoint
epoch_loaded, ckpt_metrics = load_checkpoint(model, best_checkpoint_path)
model.to(device)
model.eval()
print(f"Loaded checkpoint from epoch {epoch_loaded} (val_acc={ckpt_metrics['val_acc']:.4f})")

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_preds = np.argmax(all_probs, axis=1)

acc = compute_accuracy(all_preds, all_labels)
auc = compute_auc(all_probs, all_labels)
ece = compute_ece(all_probs, all_labels)

print(f"\n{'='*40}")
print(f"  Test Results (CIFAKE)")
print(f"{'='*40}")
print(f"  Accuracy : {acc:.4f}")
print(f"  AUC-ROC  : {auc:.4f}")
print(f"  ECE      : {ece:.4f}")
print(f"{'='*40}")

plot_reliability_diagram(all_probs, all_labels, os.path.join(PLOTS_DIR, "reliability_diagram.png"))
plot_roc_curve(all_probs, all_labels, os.path.join(PLOTS_DIR, "roc_curve.png"))
print(f"\nPlots saved to {PLOTS_DIR}/")